# 05 – Feature Engineering

Create order-level, customer-level and seller-level analytical features from the cleaned datasets. Outputs are saved in `data/cleaned/engineered/` so existing dashboard files are not overwritten.

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/Cart2Insights")
CLEAN = ROOT / "data" / "cleaned"
RAW = ROOT / "data" / "raw"

if not CLEAN.exists():
    raise FileNotFoundError("Project data/cleaned folder was not found.")

print("Cleaned data folder:", CLEAN)


In [ ]:

items = pd.read_csv(CLEAN / "order_items_cleaned.csv")
orders = pd.read_csv(CLEAN / "orders_cleaned.csv")
customers = pd.read_csv(CLEAN / "customers_cleaned.csv")


In [ ]:

# Order-level features
order_features = items.groupby("order_id").agg(
    total_order_value=("price", "sum"),
    total_freight=("freight_value", "sum"),
    item_count=("order_item_id", "count")
).reset_index()

order_features = order_features.merge(
    orders, on="order_id", how="left"
)

order_features["order_purchase_timestamp"] = pd.to_datetime(
    order_features["order_purchase_timestamp"], errors="coerce"
)
order_features["order_delivered_customer_date"] = pd.to_datetime(
    order_features["order_delivered_customer_date"], errors="coerce"
)
order_features["order_estimated_delivery_date"] = pd.to_datetime(
    order_features["order_estimated_delivery_date"], errors="coerce"
)

order_features["delivery_days"] = (
    order_features["order_delivered_customer_date"] -
    order_features["order_purchase_timestamp"]
).dt.total_seconds() / 86400

order_features["delay_days"] = (
    order_features["order_delivered_customer_date"] -
    order_features["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

order_features["is_delayed"] = order_features["delay_days"] > 0
display(order_features.head())


In [ ]:

# Customer-level features
customer_features = order_features.groupby("customer_id").agg(
    order_count=("order_id", "nunique"),
    total_spending=("total_order_value", "sum"),
    average_order_value=("total_order_value", "mean")
).reset_index()

customer_features["is_repeat_customer"] = (
    customer_features["order_count"] > 1
)

# Seller-level features
seller_features = items.groupby("seller_id").agg(
    seller_revenue=("price", "sum"),
    item_count=("order_item_id", "count"),
    order_count=("order_id", "nunique")
).reset_index()

output = CLEAN / "engineered"
output.mkdir(exist_ok=True)

order_features.to_csv(output / "orders_with_features.csv", index=False)
customer_features.to_csv(output / "customer_features.csv", index=False)
seller_features.to_csv(output / "seller_features.csv", index=False)

print("Feature files saved in:", output)
print("Customer features:", customer_features.shape)
print("Seller features:", seller_features.shape)
